# 07 — Deep Learning: Fine-tuning BERT for Sentiment Analysis
**Covers:** Tokenisation · BERT fine-tuning · Training loop with mixed precision · LR scheduling · Evaluation · Inference

---

In [ ]:
!pip install -q transformers datasets torch scikit-learn matplotlib pandas accelerate

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import load_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

%matplotlib inline

## 1. Load Data

In [ ]:
raw = load_dataset("glue", "sst2")
print(raw)

# Use a subset for faster iteration — remove [:1000] for full training
train_data = raw['train'].select(range(min(5000, len(raw['train']))))
val_data   = raw['validation']

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

## 2. Tokenisation

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'   # fast; swap for 'bert-base-uncased' or 'roberta-base'
MAX_LEN = 128
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Demo tokenisation
sample = "This movie was absolutely fantastic!"
enc = tokenizer(sample, return_tensors='pt')
print("Tokens:", tokenizer.convert_ids_to_tokens(enc['input_ids'][0]))
print("Input IDs:", enc['input_ids'][0].tolist())

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len):
        self.data = hf_dataset
        self.tok  = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc = self.tok(
            item['sentence'],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(item['label'], dtype=torch.long)
        }

train_ds = SentimentDataset(train_data, tokenizer, MAX_LEN)
val_ds   = SentimentDataset(val_data,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 3. Model Setup

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print(model.config)

## 4. Optimizer, Scheduler, Mixed Precision

In [ ]:
from torch.cuda.amp import GradScaler, autocast

EPOCHS = 3
LR = 2e-5
WARMUP_RATIO = 0.1

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=0.01, eps=1e-8
)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

scaler = GradScaler(enabled=torch.cuda.is_available())

print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

## 5. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss, preds, trues = 0, [], []
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        with autocast(enabled=torch.cuda.is_available()):
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        preds.extend(outputs.logits.argmax(-1).cpu().tolist())
        trues.extend(labels.cpu().tolist())

    return total_loss / len(loader), accuracy_score(trues, preds)

def evaluate(model, loader):
    model.eval()
    total_loss, preds, trues = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds.extend(outputs.logits.argmax(-1).cpu().tolist())
            trues.extend(labels.cpu().tolist())
    return total_loss/len(loader), accuracy_score(trues, preds), f1_score(trues, preds)

history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
best_f1, best_state = 0.0, None

for epoch in range(1, EPOCHS+1):
    tl, ta = train_epoch(model, train_loader, optimizer, scheduler, scaler)
    vl, va, vf = evaluate(model, val_loader)
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va); history['val_f1'].append(vf)
    if vf > best_f1:
        best_f1 = vf
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss={tl:.4f} Acc={ta:.4f} | Val Loss={vl:.4f} Acc={va:.4f} F1={vf:.4f}")

model.load_state_dict(best_state)
print(f"\nBest Val F1: {best_f1:.4f}")

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs = range(1, EPOCHS+1)

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train')
axes[1].plot(epochs, history['val_acc'],   'r-o', label='Val')
axes[1].set_title('Accuracy'); axes[1].legend()

axes[2].plot(epochs, history['val_f1'], 'g-o', label='Val F1')
axes[2].set_title('Val F1'); axes[2].legend()

for ax in axes: ax.set_xlabel('Epoch')
plt.suptitle(f'Fine-tuning {MODEL_NAME}', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Detailed Evaluation

In [ ]:
model.eval()
all_preds, all_trues = [], []
with torch.no_grad():
    for batch in val_loader:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        labs  = batch['labels'].to(DEVICE)
        logits = model(ids, attention_mask=mask).logits
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_trues.extend(labs.cpu().tolist())

print(classification_report(all_trues, all_preds, target_names=['Negative', 'Positive']))

## 8. Inference Pipeline

In [ ]:
from transformers import pipeline as hf_pipeline

# Convenience wrapper — uses the fine-tuned model
def predict(texts, batch_size=16):
    model.eval()
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, max_length=MAX_LEN, padding=True, truncation=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        for text, prob in zip(batch, probs):
            label = 'Positive' if prob[1] > 0.5 else 'Negative'
            results.append({'text': text, 'label': label, 'P(Pos)': f'{prob[1]*100:.1f}%'})
    return results

# Test
test_texts = [
    "One of the greatest films I have ever seen. Absolutely breathtaking.",
    "Predictable plot, wooden acting. A disappointing mess from start to finish.",
    "The cinematography is stunning but the story is slow.",
]

for r in predict(test_texts):
    print(f"[{r['label']:8s} | P(Pos)={r['P(Pos)']:>6}]  {r['text']}")

## 9. Save & Load Model

In [ ]:
SAVE_DIR = '/tmp/sentiment_bert_finetuned'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

# Reload
loaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(DEVICE)
loaded_tok   = AutoTokenizer.from_pretrained(SAVE_DIR)
print("Model loaded successfully.")

---
**Next Notebook →** `08_genai_prompt_models.ipynb`